[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/quadrics/quadrics.ipynb)

# An Ellipsoid as a Map from Planes to Points

This example constructs a three-dimensional ellipsoid from its tangent planes, recovers a tangent plane from its contact point, and transports the surface rigidly by a motor. Every tangent plane touches a smooth convex ellipsoid at one contact point, and every point of the surface has one tangent plane, so the surface is modelled as the duality map sending each plane to its pole.

In projective geometric algebra, planes are vectors (`Plane+`), points are antivectors (`Point-`), and the dual quadric `Point- <- Plane+` sums the dyads `semi_axes * (Plane & semi_axes)` of the ideal semi-axes less the dyad `center * (Plane & center)` of the center. A plane is tangent where it contains its own image, `tangent & dual(tangent) == 0`, the image of the plane at infinity is the center, `dual.solve(contact)` recovers the tangent plane, and a motor carries the map by pulling planes into the body frame and pushing points back out, `motor >> dual(motor << Plane)`.

In the homogeneous matrix notation of projective geometry, a dual quadric is written as a symmetric $4 \times 4$ matrix $Q^*$ with tangency $\boldsymbol\pi^{\mathsf T} Q^* \boldsymbol\pi = 0$ and contact point $\mathbf x = Q^* \boldsymbol\pi$, while a rigid motion $(R, \mathbf t) \in \mathrm{SE}(3)$ builds the block transformation $H = \bigl[\begin{smallmatrix} R & \mathbf t \\ \mathbf 0^{\mathsf T} & 1 \end{smallmatrix}\bigr]$, carries planes contravariantly by $\boldsymbol\pi' = H^{-\mathsf T}\boldsymbol\pi$, and conjugates the dual quadric as $Q^{*\prime} = H Q^* H^{\mathsf T}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.quadrics.quadrics import render, scenarios
from examples.util.animation import save_figure

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Motor = ga.gatype.rotor()                           # Rotor+
DualQuadric = ga.gatype((Point, Plane))             # Point- <- Plane+

## Polar reciprocity from semi-axis dyads

Each semi-axis is an ideal direction point, and its dyad `semi_axes * (Plane & semi_axes)` weights that direction by how strongly a plane faces along it; subtracting the center dyad closes the envelope of tangent planes around the center. Shifting a normal plane through the pole of infinity and sliding it outward until it contains its own image locates the support plane in that direction, and solving the map against the contact point recovers the plane.

In [ ]:
semi_axes = scenarios.semi_axes
center = scenarios.center
facing = scenarios.facing

In [ ]:
def ellipsoid(semi_axes: Point, center: Point) -> DualQuadric:
    """Sums the dyads of the ideal `semi_axes` (`[3] Point-`) less the dyad of `center` (`[] Point-`) into a dual quadric (`[] Point- <- Plane+`)."""
    # Ideal points give the directional shape; subtracting the center dyad closes the envelope of tangent planes.
    return (semi_axes * (Plane & semi_axes)).sum(axis=0) - center * (Plane & center)   # [] Point- <- Plane+


def support_plane(quadric: DualQuadric, facing: Plane) -> Plane:
    """Slides the plane facing `facing` (`[...] Plane+`) out from the pole of infinity until it contains its own image under `quadric` (`[...] Plane+`)."""
    normal = facing.normalized()                                                       # [...] Plane+
    # The center of the ellipsoid is the image of the plane at infinity.
    center = quadric(mv.w)                                                             # [] Point-
    through_center = normal - mv.w * ((normal & center) / (mv.w & center))             # [...] Plane+
    # Dividing by the center's weight makes the offset independent of the quadric's overall scale.
    offset = (-(through_center & quadric(through_center)) / (mv.w & center)).square_root()   # [...] Scalar+
    return through_center - mv.w * offset                                              # [...] Plane+


dual = ellipsoid(mv.direction(semi_axes), mv.point(center))                            # [] Point- <- Plane+
tangent = support_plane(dual, mv.vector(facing))                                       # [] Plane+
contact = dual(tangent)                                                                # [] Point-
recovered_tangent = dual.solve(contact)                                                # [] Plane+
save_figure(render.draw_polar_reciprocity(dual, tangent, contact), "quadrics_polar_reciprocity");

The translucent blue surface is ray-traced directly from the inverse point quadric around the blue center dot, while the red square patch marks the support plane touching the surface at its red contact point.

In [ ]:
print(dual.coefficient_table())

## Rigid transport of the dual quadric

A motor moves planes and points by the same sandwich product, so it transports the ellipsoid's duality map by pulling world planes back into the body frame, applying `dual` there, and pushing the resulting contact points forward into world space. Carrying the support plane by the motor and evaluating the transported quadric lands on the transported contact point.

In [ ]:
displacement = scenarios.displacement
turn_angle = scenarios.turn_angle

In [ ]:
# Each translation generator is an ideal line formed by wedging a coordinate plane with the plane at infinity.
translation = (stack((mv.xw, mv.yw, mv.zw)) * displacement).sum(axis=0)        # [] Bivector+
motor = (translation / 2).exp() * (mv.xy * (turn_angle / 2)).exp()              # [] Rotor+
world = motor >> dual(motor << Plane)                                           # [] Point- <- Plane+
world_tangent = motor >> tangent                                                # [] Plane+
world_contact = world(world_tangent)                                            # [] Point-
save_figure(
    render.draw_motor_transport(dual, tangent, contact, world, world_tangent, world_contact),
    "quadrics_motor_transport",
);

The left panel shows the reference ellipsoid and its support patch at the origin, and the right panel shows the rotated and translated quadric carrying its tangent patch and contact point together.

In [ ]:
print("center after transport:\n", world(mv.w).coefficient_table())

In [ ]:
# checks
moved_center = world(mv.w)                                                     # [] Point-
center_offset = (stack((mv.x, mv.y, mv.z)) & moved_center) / (mv.w & moved_center)   # [3] Scalar+
assert (tangent & contact).abs() < 1e-12
assert (recovered_tangent & contact).abs() < 1e-12
assert (np.abs((recovered_tangent - tangent).kernel) < 1e-12).all()
assert (world_tangent & world_contact).abs() < 1e-12
assert (np.abs((world_contact - (motor >> contact)).kernel) < 1e-11).all()
assert ((center_offset - displacement).abs() < 1e-12).all()